In [ ]:
import pandas as pd
import geopandas as gpd
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv
import rasterio
from rasterio.merge import merge
from rasterio.mask import mask
import numpy as np
from shapely.geometry import mapping
from shapely import geometry
from scipy import ndimage
from shapely import wkb
load_dotenv()

db_string = os.getenv('DB_STRING_PROD')
engine = create_engine(db_string)

def build_zones(geom):
    z0 = geom
    z1 = geom.buffer(10).difference(geom)
    z2 = geom.buffer(20).difference(geom.buffer(10))
    z3 = geom.buffer(50).difference(geom.buffer(20))
    z4 = geom.buffer(100).difference(geom.buffer(50))

    return {
        "z0": z0,
        "z1": z1,
        "z2": z2,
        "z3": z3,
        "z4": z4,
    }
    
def extract_zone_raster(src, geom, bands_idx):
    if geom.is_empty:
        return None

    out, _ = mask(
        src,
        [mapping(geom)],
        crop=True,
        nodata=src.nodata
    )

    # sélection bandes 12,13,14
    data = out[bands_idx, :, :].astype(float)

    if src.nodata is not None:
        data[data == src.nodata] = np.nan

    return data

def max_continuous_surface(data, threshold, pixel_area):
    """
    data : (H, W)
    """
    binary = data > threshold

    labeled, n = ndimage.label(binary)
    if n == 0:
        return 0.0

    sizes = ndimage.sum(binary, labeled, range(1, n + 1))
    return np.max(sizes) * pixel_area
    
def compute_stats(data, pixel_area):
    """
    data : 2D np.array (H, W) with NaNs
    pixel_area : surface of one pixel (m²)
    """

    valid = data[~np.isnan(data)]
    total_pixels = valid.size

    if total_pixels == 0:
        return {
            "mean": np.nan,
            "low_decile_mean": np.nan,
            "high_decile_mean": np.nan,
            "surf_gt_50_ratio": 0.0,
            "surf_gt_100_ratio": 0.0,
            "count_gt_50_ratio": 0.0,
            "count_gt_100_ratio": 0.0,
            "count_gt_150_ratio": 0.0,
        }

    # Quantiles
    q10 = np.percentile(valid, 10)
    q90 = np.percentile(valid, 90)

    low_decile_mean = valid[valid <= q10].mean()
    high_decile_mean = valid[valid >= q90].mean()

    # Total surface
    total_surface = total_pixels * pixel_area

    # Continuous surfaces
    surf_gt_50 = max_continuous_surface(data, 50, pixel_area)
    surf_gt_100 = max_continuous_surface(data, 100, pixel_area)

    return {
        "mean": valid.mean()/255,
        "low_decile_mean": low_decile_mean/255,
        "high_decile_mean": high_decile_mean/255,

        # normalized continuous surfaces
        "surf_gt_50_ratio": surf_gt_50 / total_surface,
        "surf_gt_100_ratio": surf_gt_100 / total_surface,

        # normalized pixel counts
        "count_gt_50_ratio": np.sum(valid > 50) / total_pixels,
        "count_gt_100_ratio": np.sum(valid > 100) / total_pixels,
        "count_gt_150_ratio": np.sum(valid > 150) / total_pixels,
    }

def build_features(gdf_targets, raster_img_path_output_list, bands_idx, pixel_area):

    """
    avec une image d entrée au format (16, H, W) : 
    - on travaille uniquement sur les bandes 12 13 et 14
    - pour chacune des 5 zones suivantes : 
        - z0 : geometry du bati
        - z1 : buffer 10m autour du bati zone du bati exclue
        - z2 : buffer de 10 a 20m
        - z3 : buffer de 20 a 50m 
    on calcule les grandeurs suivantes de chacune des couches : 
        - moyenne
        - min max
        - surface max continue au dessus de 50
        - surface max continue au dessus de 100
        - nombre de valeurs au dessus de 50
        - nombre de valeurs au dessus de 100
        - nombre de valeurs au dessus de 150
    """

    results = []
    for raster_img_path in raster_img_path_output_list:
        
        with rasterio.open(raster_img_path) as src:  
            print(f"processing raster image : {raster_img_path}")
            print(f"control crs : raster image {src.crs} - target_geometry {gdf_targets.crs}")
            
            for row in gdf_targets.itertuples():
                    print(f"processing data row {row.Index} - parcel :{row.id_parcellaire}")
                    zones = build_zones(row.geometry)
                    
                    for zone_name, zone_geom in zones.items():
                        try :
                            data = extract_zone_raster(src, zone_geom, bands_idx)
                            print(f"Data extracted for row {row.Index} zone {zone_name} ")
                        except:
                            print(f"No intersection between bat and raster segmentation source")
                            data = None
                            
                        if data is None:
                            continue

                        for i, band in enumerate(bands_idx):
                            stats = compute_stats(data[i], pixel_area)
                            print(f"Features built for : row {row.Index} - zone {zone_name} - band {band}")
                            results.append({
                                "building_id": row.Index,
                                "zone": zone_name,
                                "band": band,
                                **stats
                            })


    df_stats = pd.DataFrame(results)
    df_stats = df_stats.drop_duplicates(subset=['building_id','zone','band'], keep='first')

    metrics = [
    "mean",
    "low_decile_mean",
    "high_decile_mean",
    "surf_gt_50_ratio",
    "surf_gt_100_ratio",
    "count_gt_50_ratio",
    "count_gt_100_ratio",
    "count_gt_150_ratio",
    ]

    df_wide = (
        df_stats
        .set_index(["building_id", "zone", "band"])[metrics]
        .unstack(["zone", "band"])
    )

    # Flatten multi-index columns
    df_wide.columns = [
        f"{zone}_{band}_{metric}"
        for metric, zone, band in df_wide.columns
    ]

    df_wide = df_wide.reset_index()

    gdf_dataset = gdf_targets[['building_id','bati_buffer_50m','id_parcellaire','target_pv','target_control','observations']].merge(df_wide, how='left', on='building_id')
    return gdf_dataset

# 1. données annotées d entrée DDTM34

#### a) preparation fichier

In [ ]:
df_ = pd.read_excel("/app/data/datasets/debug/bush/20251020SyntheseControlesOLDAvecPV.ods")
df = df_[['IDU','CODE_INSEE','REFERENCES CADASTRALES','ANNEE','OBSERVATIONS','LIGNE BAS','ARBRES MALVENANTS','MISE A DISTANCE HOUPPIERS','MISE A DISTANCE CONSTRUCTION','AVERTISSEMENT','TA']]
df['id_parcellaire']=df['CODE_INSEE'].astype(str)+ '000'+ df['REFERENCES CADASTRALES']
communes_inspectees = df.CODE_INSEE.drop_duplicates().tolist()
df.head(5)

In [ ]:
df_parcels = gpd.read_postgis("select * from detections.fr_herault_parcels", geom_col='geometry',con=engine)
df_parcels.head(5)

In [ ]:
#df_['TA'] = df_.TA.replace('-',pd.NA)
#df_['AVERTISSEMENT'] = df_.AVERTISSEMENT.replace('-',pd.NA)
df_.TA.fillna(0,inplace=True)
df_.AVERTISSEMENT.fillna(0,inplace=True)
df_.groupby(['CODE_INSEE','COMMUNE','TA','AVERTISSEMENT']).size().reset_index().sort_values(ascending=False, by=['COMMUNE',0]).head(50)
df_

traitées : velieux, berlou, montoulier, boissiere, pierrerue

exhaustives : BOISSIERE 34035,  34201	PIERRERUE, 34113	GIGEAN (a vérifier si me semble ok), 34030	BERLOU, 34021	BABEAU BOULDOUX, 34314	TRIADOU, 34006	AIGNE, 
34162	MONTAGNAC, 34326	VELIEUX, 34137	LIAUSSON, 34170	MONTOULIERS, 34189	OLONZAC, 34190	OUPIA

communes a infos a exclure car pv non : 

les autres communes peuvent etre utilisées en target positives controle et pv mais pas en 0.


NON CONFORME : si avertissement ou TA a 1
CONFORME : les autres
si on part de la bd topo, les ruines ne sont pas a controler

### b) extraction des données annotées

In [ ]:
commune_annotated = {'name':'boissiere', 'geozone_code': 34035,'geozone_id': 272,'result_segmentation_files':  ['/app/runs/aigle_aerial_yolov_2024_boissiere_34035_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_209.tif',
                                '/app/runs/aigle_aerial_yolov_2024_boissiere_34035_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_210.tif',
                               '/app/runs/aigle_aerial_yolov_2024_boissiere_34035_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_222.tif']}

working_geozone_code = commune_annotated['geozone_code']
working_geozone_id = commune_annotated['geozone_id']
working_result_segmentation_files = commune_annotated['result_segmentation_files']

In [ ]:
# filter on a single commune for study
df_parcels = df_parcels[df_parcels.commune_id==working_geozone_id]
df_data = df[df.CODE_INSEE==working_geozone_code]
print(f"Nb parcels in commune : {len(df_parcels)}")
print(f"Nb parcels controlled : {len(df_data)}")
print(f"Nb parcels with PV : {len(df_data[(df_data.TA==1) | (df_data.AVERTISSEMENT==1)])}")

In [ ]:
# format target column
df_data['target_control'] = 1
df_data['target_pv'] = 0
df_data.loc[(df_data.TA==1) | (df_data.AVERTISSEMENT==1), 'target_pv'] = 1

print(f"Nb parcels in commune : {len(df_parcels)}")
print(f"Nb parcels controlled : {len(df_data[df_data.target_control==1])}")
print(f"Nb parcels with PV : {len(df_data[df_data.target_pv==1])}")

In [ ]:
# format dataset
df_data = df_data.merge(df_parcels, how='right', on='id_parcellaire')

df_data = df_data[['commune_id','id_parcellaire','geometry','target_pv','target_control','OBSERVATIONS']]
df_data.rename(columns={'OBSERVATIONS': 'observations'}, inplace=True)
df_data.target_pv.replace('-',pd.NA, inplace=True)
df_data.target_pv.fillna(0,inplace=True)
df_data.target_control.fillna(0,inplace=True)
df_data.target_pv.astype('int32')
df_data.target_control.astype('int32')
df_data.drop_duplicates(subset='id_parcellaire', keep='first')

" pour la suite (c & d) on va integrer les bd topo et les zones de débroussaillage pour filtrer un dataset pertinent"

##### c) visualisation des parcelles controllées et verbalisées

In [ ]:
gdf_data = gpd.GeoDataFrame(df_data)
gdf_data.to_file("/app/data/datasets/debug/bush/couche_controles_boissiere.gpkg",driver='GPKG')

In [ ]:
gdf_data_pv = gdf_data[gdf_data.target_pv==1]
gdf_data_pv.to_file("/app/data/datasets/debug/bush/couche_controles_boissiere_avec_pv_etabli.gpkg",driver='GPKG')
gdf_data_pv

# d) récuperation de la bd topo

In [ ]:
db_topo_gdf_ = gpd.read_file("/app/data/aigle_aerial_yolov_2024_berlou_34030_v1.1/db-cache/BATIMENT.shp")
db_topo_gdf_.rename(columns={'DATE_CREAT':'date_creat'},inplace=True)
_drop_z = lambda geom: wkb.loads(wkb.dumps(geom, output_dimension=2))
db_topo_gdf_['geometry'] = db_topo_gdf_['geometry'].apply(_drop_z)
db_topo_gdf = db_topo_gdf_[['date_creat','geometry']]        
db_topo_gdf['date_creat'] = pd.to_datetime(db_topo_gdf.date_creat)

In [ ]:
db_topo_gdf_.columns

In [ ]:
db_topo_gdf.to_crs("EPSG:4326",inplace=True)

In [ ]:

df_topo_in_geozone = gpd.sjoin(db_topo_gdf, df_parcels, how='left',predicate='intersects')
df_topo_in_geozone = df_topo_in_geozone[~df_topo_in_geozone.index_right.isna()]
df_topo_in_geozone =df_topo_in_geozone[['geometry','date_creat','commune_id','id_parcellaire']]

df_topo_in_geozone.to_file("/app/data/datasets/debug/bush/couche_db_topo_geozone.gpkg",driver='GPKG')

In [ ]:
gdf_targets = gpd.sjoin(df_topo_in_geozone, gpd.GeoDataFrame(df_data[['geometry','target_pv','target_control','observations']], geometry='geometry'),how='left',predicate='within')
gdf_targets.drop(columns=['index_right'],inplace=True)
gdf_targets.target_pv.fillna(0,inplace=True)
gdf_targets.target_control.fillna(0,inplace=True)
gdf_targets

In [ ]:
gdf_targets.drop_duplicates(subset='geometry',keep='first', inplace=True)

# e. filtrer sur la zone risque feux 

In [ ]:
df_risk_bush = gpd.read_file('/app/data/datasets/debug/bush/Debroussaillement_light.gpkg')
df_risk_bush

In [ ]:
# a verifier sur une commune partiellement dans la zone risque
gdf_targets = gpd.sjoin(gdf_targets, df_risk_bush[['geometry']].to_crs('EPSG:4326'), how='left', predicate='intersects').drop(columns=['index_right'])
gdf_targets

In [ ]:
gdf_targets.to_crs('EPSG:2154',inplace=True)
gdf_targets['bati_buffer_50m'] = gdf_targets.geometry.buffer(50)

gdf_targets = gdf_targets.reset_index().rename(columns={'index':'building_id'})
gdf_targets

In [ ]:
gdf_targets[['bati_buffer_50m','target_pv','target_control']].set_geometry('bati_buffer_50m').to_file('/app/data/datasets/debug/bush/targets_berlou.gpkg')

### f) zone filtre d'interet DDT

In [ ]:
complete_insee_code_list = ['34030','34170','34326','34035','34201']
partial_insee_code_list = ['34028']

In [ ]:
df_zones_r = gpd.read_postgis("select * from detections.n_dfci_old50m_s_034_ilots", geom_col='geom',con=engine)
df_zones_r

In [ ]:
df_zones_r.to_file('/app/data/datasets/debug/bush/zone_old.gpkg', driver='GPKG')

In [ ]:
# on recupere toutes les zones ilots remontées par la ddt
# on recupere tous les targets de controle de l equipe ddt
# on controle que tout match correctement
# on conserve les geometries des zones old
# on assigne les targets du modele dol on cree un dataset ilots v1

In [ ]:
df_ = pd.read_excel("/app/data/datasets/debug/bush/20251020SyntheseControlesOLDAvecPV.ods")
df = df_[['IDU','CODE_INSEE','COMMUNE','REFERENCES CADASTRALES','ANNEE','OBSERVATIONS','LIGNE BAS','ARBRES MALVENANTS','MISE A DISTANCE HOUPPIERS','MISE A DISTANCE CONSTRUCTION','AVERTISSEMENT','TA']]
df['id_parcellaire']=df['CODE_INSEE'].astype(str)+ '000'+ df['REFERENCES CADASTRALES']
# format and clean data
df['TA'] = df.TA.fillna(0)
df['AVERTISSEMENT'] = df.AVERTISSEMENT.fillna(0)

# format target column
df['target_control'] = 1
df['target_pv'] = 0
df.loc[(df.TA==1) | (df.AVERTISSEMENT==1), 'target_pv'] = 1


In [ ]:
df_parcels = gpd.read_postgis("select * from detections.fr_herault_parcels", geom_col='geometry',con=engine)
df_parcels['insee_code'] = df_parcels['id_parcellaire'].str[:5]


In [ ]:
df_data = df.merge(df_parcels, how='left', on='id_parcellaire')

df_data = df_data[['commune_id','CODE_INSEE','id_parcellaire','geometry','target_pv','target_control','OBSERVATIONS']]
df_data.rename(columns={'OBSERVATIONS': 'observations', 'CODE_INSEE': 'code_insee'}, inplace=True)
df_data.target_pv.replace('-',pd.NA, inplace=True)
df_data.target_pv.fillna(0,inplace=True)
df_data.target_control.fillna(0,inplace=True)
df_data.target_pv.astype('int32')
df_data.target_control.astype('int32')
df_data.drop_duplicates(subset='id_parcellaire', keep='first')

In [ ]:
gdf_data_cplt = gpd.GeoDataFrame(df_data[df_data.code_insee.astype('str').isin(complete_insee_code_list)],geometry='geometry')
gdf_data_ptl = gpd.GeoDataFrame(df_data[df_data.code_insee.astype('str').isin(partial_insee_code_list)],geometry='geometry')
gdf_data_cplt.head(5)

In [ ]:
gdf_zones_r_cplt = df_zones_r[df_zones_r.insee_com.astype('str').isin(complete_insee_code_list)].to_crs('EPSG:4326')
gdf_zones_r_ptl = df_zones_r[df_zones_r.insee_com.astype('str').isin(partial_insee_code_list)].to_crs('EPSG:4326')
gdf_zones_r_cplt.head(5)

In [ ]:
# deal with completes zones
s = gpd.overlay(df_parcels, gdf_zones_r_cplt, how = 'intersection') 
s['area_ov'] = s.geometry.area
gb = s.groupby(['id_ilot','id_parcellaire'])[['area_ov']].max()
mapping_ilot_parcel = gb.reset_index().sort_values(by=['id_ilot','area_ov'], ascending=False).drop(columns='area_ov')
mapping_ilot_parcel

gdf_data = gdf_zones_r_cplt.merge(mapping_ilot_parcel, how = 'left', on='id_ilot')
gdf_data['id_parcellaire']=gdf_data['id_parcellaire'].astype('string')
gdf_data_cplt['id_parcellaire']=gdf_data_cplt['id_parcellaire'].astype('string')
gdf_data = gdf_data.merge(gdf_data_cplt, how = 'left', on='id_parcellaire')
gdf_data.target_pv.fillna(0,inplace=True)
gdf_data.target_control.fillna(0,inplace=True)

gdf_zone_data_cplt = gdf_data.groupby(['id_ilot','code_ilot','compte_communal','insee_com','geom']).aggregate({'target_control' : np.sum, 'target_pv' : np.sum}).reset_index()
gdf_zone_data_cplt.loc[gdf_zone_data_cplt.target_control>1,'target_control']=1
gdf_zone_data_cplt.loc[gdf_zone_data_cplt.target_pv>1,'target_pv']=1
gdf_zone_data_cplt = gpd.GeoDataFrame(gdf_zone_data_cplt,geometry='geom')
gdf_zone_data_cplt.head(5)

In [ ]:
s = gpd.overlay(gdf_data_ptl, gdf_zones_r_ptl, how = 'intersection') 
s['area_ov'] = s.geometry.area
gb = s.groupby(['id_ilot','id_parcellaire'])[['area_ov']].max()
mapping_ilot_parcel = gb.reset_index().sort_values(by=['id_ilot','area_ov'], ascending=False).drop(columns='area_ov')
mapping_ilot_parcel

gdf_data = gdf_zones_r_ptl.merge(mapping_ilot_parcel, how = 'left', on='id_ilot')
gdf_data['id_parcellaire']=gdf_data['id_parcellaire'].astype('string')
gdf_data_ptl['id_parcellaire']=gdf_data_ptl['id_parcellaire'].astype('string')
gdf_data = gdf_data.merge(gdf_data_ptl, how = 'left', on='id_parcellaire')
gdf_data.target_pv.fillna(0,inplace=True)
gdf_data.target_control.fillna(0,inplace=True)

gdf_zone_data_ptl = gdf_data.groupby(['id_ilot','code_ilot','compte_communal','insee_com','geom']).aggregate({'target_control' : np.sum, 'target_pv' : np.sum}).reset_index()
gdf_zone_data_ptl.loc[gdf_zone_data_ptl.target_control>1,'target_control']=1
gdf_zone_data_ptl.loc[gdf_zone_data_ptl.target_pv>1,'target_pv']=1
gdf_zone_data_ptl = gpd.GeoDataFrame(gdf_zone_data_ptl,geometry='geom')

# filter data only on controled samples
gdf_zone_data_ptl = gdf_zone_data_ptl[gdf_zone_data_ptl.target_control==1]
gdf_zone_data_ptl.head(5)

In [ ]:
# match the corresponding raster
communes_annotated_clean = [
    {'name':'berlou', 'geozone_code': 34030,'geozone_id': 87,'result_segmentation_files':  ['/app/runs/aigle_aerial_yolov_2024_berlou_34030_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_52.tif',
                               '/app/runs/aigle_aerial_yolov_2024_berlou_34030_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_61.tif']},
    {'name':'montouliers', 'geozone_code': 34170,'geozone_id': 183,'result_segmentation_files':  ['/app/runs/aigle_aerial_yolov_2024_montouliers_34170_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_48.tif',
                               '/app/runs/aigle_aerial_yolov_2024_montouliers_34170_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_49.tif']},
    {'name':'velieux', 'geozone_code': 34326,'geozone_id': 156,'result_segmentation_files':  ['/app/runs/aigle_aerial_yolov_2024_velieux_34326_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_23.tif',
                               '/app/runs/aigle_aerial_yolov_2024_velieux_34326_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_24.tif',
                               '/app/runs/aigle_aerial_yolov_2024_velieux_34326_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_33.tif',
                               '/app/runs/aigle_aerial_yolov_2024_velieux_34326_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_34.tif']},
     {'name':'boissiere', 'geozone_code': 34035,'geozone_id': 272,'result_segmentation_files':  ['/app/runs/aigle_aerial_yolov_2024_boissiere_34035_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_209.tif',
                                '/app/runs/aigle_aerial_yolov_2024_boissiere_34035_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_210.tif',
                               '/app/runs/aigle_aerial_yolov_2024_boissiere_34035_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_222.tif',
                               '/app/runs/aigle_aerial_yolov_2024_boissiere_34035_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_223.tif']},
     {'name':'pierrerue', 'geozone_code': 34201,'geozone_id': 270,'result_segmentation_files':  ['/app/runs/aigle_aerial_yolov_2024_pierrerue_34201_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_50.tif',
                                                                                                 '/app/runs/aigle_aerial_yolov_2024_pierrerue_34201_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_51.tif',
                                                                                                 '/app/runs/aigle_aerial_yolov_2024_pierrerue_34201_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_59.tif',
                                                                                                 '/app/runs/aigle_aerial_yolov_2024_pierrerue_34201_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_60.tif',
                                                                                                 '/app/runs/aigle_aerial_yolov_2024_pierrerue_34201_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_70.tif',
                                                                                                 '/app/runs/aigle_aerial_yolov_2024_pierrerue_34201_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_71.tif']}
     ]

communes_annotated_partially = [
    {'name':'bedarieux', 'geozone_code': 34028,'geozone_id': 3,'result_segmentation_files':  ['/app/runs/aigle_aerial_yolov_2024_bedarieux_34028_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_88.tif',
        '/app/runs/aigle_aerial_yolov_2024_bedarieux_34028_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_89.tif',
        '/app/runs/aigle_aerial_yolov_2024_bedarieux_34028_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_102.tif',
        '/app/runs/aigle_aerial_yolov_2024_bedarieux_34028_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_103.tif',
        '/app/runs/aigle_aerial_yolov_2024_bedarieux_34028_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_117.tif',
        '/app/runs/aigle_aerial_yolov_2024_bedarieux_34028_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_118.tif']}
]
communes_annotated = communes_annotated_clean + communes_annotated_partially
imgs_bounds = []
for comm in communes_annotated :
    for img_path in comm['result_segmentation_files']:
        with rasterio.open(img_path) as src :
            bbox = src.bounds
            bbox_polygon = geometry.box(*bbox)
            print(bbox_polygon)
            imgs_bounds.append([img_path, bbox_polygon])

gdf_img = gpd.GeoDataFrame(data= imgs_bounds, columns=['image_path','geometry'], geometry='geometry', crs='EPSG:2154')
gdf_img.to_crs('EPSG:4326',inplace=True)

In [ ]:
gdf_img.drop_duplicates(subset='image_path')

In [ ]:
gdf_zone_data = pd.concat([gdf_zone_data_ptl, gdf_zone_data_cplt])

gdf_zone_data = gpd.sjoin(gdf_img,gdf_zone_data, how='right', predicate='intersects').drop(columns='index_left')
gdf_zone_data


In [ ]:
find_double_img = gdf_zone_data.groupby('code_ilot').size().reset_index().sort_values(by=0, ascending=False).rename(columns={0:'nb_src'})
ilots_borders = find_double_img[find_double_img['nb_src']>1].code_ilot.tolist()
gdf_zone_data = gdf_zone_data[~gdf_zone_data.code_ilot.isin(ilots_borders)]


In [ ]:
gdf_zone_data[gdf_zone_data.image_path.isna()]

In [ ]:
gdf_zone_data[~(gdf_zone_data.image_path.isna())].to_file('/app/data/datasets/debug/bush/target_dol_zones_v1.gpkg',driver='GPKG')
gdf_zone_data.groupby(['insee_com','target_control', 'target_pv']).size().reset_index()
#gdf_zone_data

# 2. Feature eng statistic methods

on construit autour de chaque batiment un buffer de 50m, au sein de cette zone, on cree 4 sous zones : 
- z0 : la zone bati
- z1 : la zone bordure bati - distance 10m du bati
- z2 : distance 10m du bati - distance 20m du bati
- z3 : distance 20m du bati - distance 50m du bati
au sein de chacune de ces zones, on extrait les 3 couches de segmentation feuillus, epineux, vegetation au sol et on calcule des indicateurs statistiques :
- valeur moyenne du 1er decile normalisé entre 0 et 1
- valeur moyenne du 10eme decile normalisé entre 0 et 1
- pourcentage de surface max continue au dessus de 50
- surface max continue au dessus de 100
- nombre de valeurs au dessus de 50
- nombre de valeurs au dessus de 100
- nombre de valeurs au dessus de 150

### a) recuperer les resultats de segmentation des zones associées a chaque batiment

In [ ]:
# features in class prob format
"""
            12 : 'deciduous'
            13 : 'coniferous'
            14 : 'brushwood'
"""
raster_img_path_output_list = ['/app/runs/aigle_aerial_yolov_2024_berlou_34030_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_52.tif',
                               '/app/runs/aigle_aerial_yolov_2024_berlou_34030_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_61.tif']
raster_img_path_output = '/app/runs/aigle_aerial_yolov_2024_berlou_34030_v1.1/berlou_2024_AERIAL_LABEL-COSIA_class-prob.tif'


### b) feature engineering 

In [ ]:
# build features 
"""
avec une image d entrée au format (16, H, W) : 
- on travaille uniquement sur les bandes 12 13 et 14
- pour chacune des 5 zones suivantes : 
    - z0 : geometry du bati
    - z1 : buffer 10m autour du bati zone du bati exclue
    - z2 : buffer de 10 a 20m
    - z3 : buffer de 20 a 50m 
    - z4 : buffer de 50 a 100m
  on calcule les grandeurs suivantes de chacune des couches : 
      - moyenne
      - min max
      - surface max continue au dessus de 50
      - surface max continue au dessus de 100
      - nombre de valeurs au dessus de 50
      - nombre de valeurs au dessus de 100
      - nombre de valeurs au dessus de 150
"""
bands_idx = [12,13,14]
pixel_area = 0.2*0.2
results = []
for raster_img_path in raster_img_path_output_list:
    with rasterio.open(raster_img_path) as src:  
        for row in gdf_targets.itertuples():
                print(f"processing data row {row.Index} - parcel :{row.id_parcellaire}")
                zones = build_zones(row.geometry)
                
                for zone_name, zone_geom in zones.items():
                    try :
                        data = extract_zone_raster(src, zone_geom, bands_idx)
                        print(f"Data extracted for row {row.Index} zone {zone_name} ")
                    except:
                        print(f"No intersection between bat and raster segmentation source")
                        data = None
                        
                    if data is None:
                        continue

                    for i, band in enumerate(bands_idx):
                        stats = compute_stats(data[i], pixel_area)
                        print(f"Features built for : row {row.Index} - zone {zone_name} - band {band}")
                        results.append({
                            "building_id": row.Index,
                            "zone": zone_name,
                            "band": band,
                            **stats
                        })


df_stats = pd.DataFrame(results)
df_stats = df_stats.drop_duplicates(subset=['building_id','zone','band'], keep='first')

df_stats

In [ ]:
metrics = [
    "mean",
    "low_decile_mean",
    "high_decile_mean",
    "surf_gt_50_ratio",
    "surf_gt_100_ratio",
    "count_gt_50_ratio",
    "count_gt_100_ratio",
    "count_gt_150_ratio",
]

df_wide = (
    df_stats
    .set_index(["building_id", "zone", "band"])[metrics]
    .unstack(["zone", "band"])
)

# Flatten multi-index columns
df_wide.columns = [
    f"{zone}_{band}_{metric}"
    for metric, zone, band in df_wide.columns
]

df_wide = df_wide.reset_index()
df_wide

In [ ]:
gdf_dataset = gdf_targets[['building_id','bati_buffer_50m','id_parcellaire','target_pv','target_control','observations']].merge(df_wide, how='left', on='building_id')
gdf_dataset.head(5)

### c) feature engineering : solution patching reduction resolution et construction d'une img d informations

requiert de travailler sur des superficies de taille fixe, il faut donc generer le carré le plus petit contenant la zone précedente, puis rationaliser la dimension a une taille fixe definie genre 512*512
puis generer le format d entrée via de la réduction

# 3. Aggregation des traitements proposés et construction du dataset

traitées : velieux, berlou, montoulier, boissiere, pierrerue

exhaustives : BOISSIERE 34035,  34201	PIERRERUE, 34113	GIGEAN (a vérifier si me semble ok), 34030	BERLOU, 34021	BABEAU BOULDOUX, 34314	TRIADOU, 34006	AIGNE, 
34162	MONTAGNAC, 34326	VELIEUX, 34137	LIAUSSON, 34170	MONTOULIERS, 34189	OLONZAC, 34190	OUPIA

communes a infos a exclure car pv non : 

les autres communes peuvent etre utilisées en trget positives controle et pv mais pas en 0.


NON CONFORME : si avertissement ou TA a 1
CONFORME : les autresm
si on part de la bd topo, les ruines ne sont pas a controler

In [ ]:
bands_idx = [12,13,14]
pixel_area = 0.2*0.2

In [ ]:
"""
communes_annotated_clean = [
    {'name':'berlou', 'geozone_code': 34030,'geozone_id': 87,'result_segmentation_files':  ['/app/runs/aigle_aerial_yolov_2024_berlou_34030_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_52.tif',
                               '/app/runs/aigle_aerial_yolov_2024_berlou_34030_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_61.tif']}]
"""
communes_annotated_clean = [
    {'name':'berlou', 'geozone_code': 34030,'geozone_id': 87,'result_segmentation_files':  ['/app/runs/aigle_aerial_yolov_2024_berlou_34030_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_52.tif',
                               '/app/runs/aigle_aerial_yolov_2024_berlou_34030_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_61.tif']},
    {'name':'montouliers', 'geozone_code': 34170,'geozone_id': 183,'result_segmentation_files':  ['/app/runs/aigle_aerial_yolov_2024_montouliers_34170_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_48.tif',
                               '/app/runs/aigle_aerial_yolov_2024_montouliers_34170_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_49.tif']},
    {'name':'velieux', 'geozone_code': 34326,'geozone_id': 156,'result_segmentation_files':  ['/app/runs/aigle_aerial_yolov_2024_velieux_34326_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_23.tif',
                               '/app/runs/aigle_aerial_yolov_2024_velieux_34326_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_24.tif']},
     {'name':'boissiere', 'geozone_code': 34035,'geozone_id': 272,'result_segmentation_files':  ['/app/runs/aigle_aerial_yolov_2024_boissiere_34035_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_209.tif',
                                '/app/runs/aigle_aerial_yolov_2024_boissiere_34035_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_210.tif',
                               '/app/runs/aigle_aerial_yolov_2024_boissiere_34035_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_222.tif']},
     {'name':'pierrerue', 'geozone_code': 34201,'geozone_id': 270,'result_segmentation_files':  ['/app/runs/aigle_aerial_yolov_2024_pierrerue_34201_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_50.tif',
                                                                                                 '/app/runs/aigle_aerial_yolov_2024_pierrerue_34201_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_51.tif',
                                                                                                 '/app/runs/aigle_aerial_yolov_2024_pierrerue_34201_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_59.tif',
                                                                                                 '/app/runs/aigle_aerial_yolov_2024_pierrerue_34201_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_60.tif',
                                                                                                 '/app/runs/aigle_aerial_yolov_2024_pierrerue_34201_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_70.tif',
                                                                                                 '/app/runs/aigle_aerial_yolov_2024_pierrerue_34201_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_71.tif']}
     ]

communes_annotated_partially = [
    {'name':'bedarieux', 'geozone_code': 34028,'geozone_id': 3,'result_segmentation_files':  ['/app/runs/',
                               '/app/runs/']}
]

In [ ]:
df_zones_r = gpd.read_postgis("select * from detections.n_dfci_old50m_s_034_ilots", geom_col='geom',con=engine)

df_parcels = gpd.read_postgis("select * from detections.fr_herault_parcels", geom_col='geometry',con=engine)

df_risk_bush = gpd.read_file('/app/data/datasets/debug/bush/Debroussaillement_light.gpkg')

db_topo_gdf_ = gpd.read_file("/app/data/aigle_aerial_yolov_2024_berlou_34030_v1.1/db-cache/BATIMENT.shp")
db_topo_gdf_.rename(columns={'DATE_CREAT':'date_creat'},inplace=True)
_drop_z = lambda geom: wkb.loads(wkb.dumps(geom, output_dimension=2))
db_topo_gdf_['geometry'] = db_topo_gdf_['geometry'].apply(_drop_z)
db_topo_gdf = db_topo_gdf_[['date_creat','geometry']]        
db_topo_gdf['date_creat'] = pd.to_datetime(db_topo_gdf.date_creat)

df_ = pd.read_excel("/app/data/datasets/debug/bush/20251020SyntheseControlesOLDAvecPV.ods")
df = df_[['IDU','CODE_INSEE','COMMUNE','REFERENCES CADASTRALES','ANNEE','OBSERVATIONS','LIGNE BAS','ARBRES MALVENANTS','MISE A DISTANCE HOUPPIERS','MISE A DISTANCE CONSTRUCTION','AVERTISSEMENT','TA']]
df['id_parcellaire']=df['CODE_INSEE'].astype(str)+ '000'+ df['REFERENCES CADASTRALES']

In [ ]:
# format and clean data
df['TA'] = df.TA.fillna(0)
df['AVERTISSEMENT'] = df.AVERTISSEMENT.fillna(0)

# format target column
df['target_control'] = 1
df['target_pv'] = 0
df.loc[(df.TA==1) | (df.AVERTISSEMENT==1), 'target_pv'] = 1

# convert to app crs
db_topo_gdf.to_crs("EPSG:4326",inplace=True)

In [ ]:
gdf_datasets_list = []
for commune_annotated in communes_annotated_clean:
    print(f"processing geozone : {commune_annotated['name']}")
    working_geozone_id = commune_annotated['geozone_id']
    working_geozone_code = commune_annotated['geozone_code']
    working_raster_img_path_output_list = commune_annotated['result_segmentation_files']
    # pour chaque commune annoté exhaustivement :

    # on extrait les controles
    # filter on a single commune for study
    df_parcels_geozone = df_parcels[df_parcels.commune_id==working_geozone_id]
    df_data = df[df.CODE_INSEE==working_geozone_code]

    print(f"raw input data - Nb parcels in commune : {len(df_parcels_geozone)}")
    print(f"raw input data - Nb parcels controlled : {len(df_data[df_data.target_control==1])}")
    print(f"raw input data - Nb parcels with PV : {len(df_data[df_data.target_pv==1])}")

    # format dataset from ddtm controls inputs
    df_data = df_data.merge(df_parcels_geozone, how='right', on='id_parcellaire')

    df_data = df_data[['commune_id','id_parcellaire','geometry','target_pv','target_control','OBSERVATIONS']]
    df_data.rename(columns={'OBSERVATIONS': 'observations'}, inplace=True)
    df_data['target_pv'] = df_data.target_pv.replace('-',pd.NA)
    df_data['target_pv'] = df_data.target_pv.fillna(0)
    df_data['target_control'] = df_data.target_control.fillna(0)
    df_data['target_pv'] = df_data.target_pv.astype('int32')
    df_data['target_control'] = df_data.target_control.astype('int32')
    df_data.drop_duplicates(subset='id_parcellaire', keep='first')

    print(f"formatted data - Nb bati area controlled : {len(df_data[df_data.target_control==1])}")
    print(f"formatted data - Nb  bati area with PV : {len(df_data[df_data.target_pv==1])}")

    print(f"adding not controled know samples from geozone...")
    # add all volontarly non controlled samples from db_topo
    df_topo_in_geozone = gpd.sjoin(db_topo_gdf, df_parcels_geozone, how='left',predicate='intersects')
    df_topo_in_geozone = df_topo_in_geozone[~df_topo_in_geozone.index_right.isna()]
    df_topo_in_geozone =df_topo_in_geozone[['geometry','date_creat','commune_id','id_parcellaire']]

    gdf_targets = gpd.sjoin(df_topo_in_geozone, gpd.GeoDataFrame(df_data[['geometry','target_pv','target_control','observations']], geometry='geometry'),how='left',predicate='within')
    gdf_targets.drop(columns=['index_right'],inplace=True)
    gdf_targets['target_pv'] = gdf_targets.target_pv.fillna(0)
    gdf_targets['target_control'] = gdf_targets.target_control.fillna(0)

    print(f"formatted data - Nb bati area controlled : {len(gdf_targets[gdf_targets.target_control==1])}")
    print(f"formatted data - Nb bati area with PV : {len(gdf_targets[gdf_targets.target_pv==1])}")
    print(f"formatted data - Nb bati area conform not controlled : {len(gdf_targets[gdf_targets.target_control==0])}")

    # filter on la zone risque OLD
    gdf_targets = gpd.sjoin(gdf_targets, df_risk_bush[['geometry']].to_crs('EPSG:4326'), how='left', predicate='intersects').drop(columns=['index_right'])

    # construire la geometry de la zone en OLD autour du bati de chaque target
    gdf_targets.to_crs('EPSG:2154',inplace=True)
    gdf_targets['bati_buffer_50m'] = gdf_targets.geometry.buffer(50)
    gdf_targets = gdf_targets.reset_index().rename(columns={'index':'building_id'})

    print(f"target data - Nb bati area controlled : {len(gdf_targets[gdf_targets.target_control==1])}")
    print(f"target data - Nb bati area with PV : {len(gdf_targets[gdf_targets.target_pv==1])}")
    print(f"target data - Nb bati area conform not controlled : {len(gdf_targets[gdf_targets.target_control==0])}")

    # construire les features associés aux targets
    # build features
    print(f"building features for target data ...")
    print()
    gdf_dataset = build_features(gdf_targets, working_raster_img_path_output_list, bands_idx, pixel_area)
    gdf_datasets_list.append(gdf_dataset)
    print(f"dataset - Nb target controlled : {len(gdf_dataset[gdf_dataset.target_control==1])}")
    print(f"dataset - Nb target with PV : {len(gdf_dataset[gdf_dataset.target_pv==1])}")
    print(f"dataset - Nb target conform not controlled : {len(gdf_dataset[gdf_dataset.target_control==0])}")

df_dataset_part_communes_clean = pd.concat(gdf_datasets_list)

In [ ]:
gdf_dataset = gpd.read_file("/app/data/datasets/debug/bush/dataset_old_v1.gpkg")
gdf_dataset['geozone_code'] = gdf_dataset['id_parcellaire'].str[:5]
gdf_dataset.describe()
gdf_dataset.groupby(['geozone_code','target_control', 'target_pv']).size().reset_index()